# 0. Setup & Import

In [68]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("../src")

from cleaning import (
    load_selected_features,
    subset_selected_features,
    check_duplicates,
    check_exact_duplicate_columns,
    fix_days_employed_anomaly,
    fix_hidden_missing_placeholders,
    impute_gender_and_family_status,
    impute_housing_group,
    impute_ext_source,
    impute_credit_bureau_inquiries,
    impute_occupation_type,
    impute_organization_type,
    create_log_features,
    consolidate_rare_categories,
    ensure_near_zero_variance_dtype,
    impute_remaining_missing,
    validate_no_missing,
    validate_dtypes_and_categories,
)

# 1. Load Data Mentah & Daftar 91 Fitur Terpilih

Membaca data mentah (122 kolom) dan daftar 91 fitur hasil seleksi 01_Eda.ipynb. Ini wajib jadi langkah pertama karena data yang ada masih mentah, belum difilter.

In [69]:
RAW_DATA_PATH = "../data/raw/application_train.csv"
SELECTED_FEATURES_PATH = "../data/processed/selected_features_eda.csv"

df_raw = pd.read_csv(RAW_DATA_PATH)
selected_features = load_selected_features(SELECTED_FEATURES_PATH)

print(f"Shape data mentah   : {df_raw.shape}")
print(f"Jumlah fitur terpilih: {len(selected_features)}")

Shape data mentah   : (307511, 122)
Jumlah fitur terpilih: 91


# 2. Subset ke 91 Fitur + ID + TARGET

Memfilter dataframe mentah supaya hanya menyisakan 91 fitur terpilih ditambah SK_ID_CURR dan TARGET.

In [70]:
df = subset_selected_features(df_raw, selected_features)
print(f"Shape setelah subset: {df.shape}")
df.head()

Shape setelah subset: (307511, 93)


,SK_ID_CURR,TARGET,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,DAYS_EMPLOYED,AMT_GOODS_PRICE,DAYS_BIRTH,OCCUPATION_TYPE,ORGANIZATION_TYPE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,DAYS_LAST_PHONE_CHANGE,AMT_CREDIT,CODE_GENDER,FLOORSMAX_AVG,DAYS_ID_PUBLISH,FLOORSMAX_MEDI,FLOORSMAX_MODE,TOTALAREA_MODE,REGION_POPULATION_RELATIVE,LIVINGAREA_AVG,LIVINGAREA_MEDI,LIVINGAREA_MODE,APARTMENTS_AVG,APARTMENTS_MEDI,APARTMENTS_MODE,REGION_RATING_CLIENT_W_CITY,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MODE,ENTRANCES_AVG,REGION_RATING_CLIENT,ENTRANCES_MEDI,WALLSMATERIAL_MODE,AMT_ANNUITY,DAYS_REGISTRATION,ENTRANCES_MODE,EMERGENCYSTATE_MODE,HOUSETYPE_MODE,NAME_FAMILY_STATUS,AMT_REQ_CREDIT_BUREAU_YEAR,NAME_HOUSING_TYPE,NAME_CONTRACT_TYPE,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_INCOME_TOTAL,HOUR_APPR_PROCESS_START,DEF_30_CNT_SOCIAL_CIRCLE,FLAG_OWN_CAR,CNT_FAM_MEMBERS,OBS_60_CNT_SOCIAL_CIRCLE,OBS_30_CNT_SOCIAL_CIRCLE,NAME_TYPE_SUITE,DEF_60_CNT_SOCIAL_CIRCLE,CNT_CHILDREN,WEEKDAY_APPR_PROCESS_START,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMP_PHONE,FLAG_MOBIL,FLAG_EMAIL,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_5,FLAG_DOCUMENT_4,FLAG_DOCUMENT_3,FLAG_DOCUMENT_6,FLAG_DOCUMENT_21,FLAG_DOCUMENT_19,FLAG_DOCUMENT_2,FLAG_DOCUMENT_20,FLAG_DOCUMENT_13,FLAG_DOCUMENT_12,FLAG_DOCUMENT_11,FLAG_DOCUMENT_18,FLAG_DOCUMENT_17,FLAG_DOCUMENT_16,FLAG_DOCUMENT_15,FLAG_DOCUMENT_14,FLAG_CONT_MOBILE,FLAG_DOCUMENT_10,LIVE_REGION_NOT_WORK_REGION,LIVE_CITY_NOT_WORK_CITY,REG_CITY_NOT_WORK_CITY,REG_CITY_NOT_LIVE_CITY,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION
0,100002,1,0.139376,0.262949,0.083037,-637,351000.0,-9461,Laborers,Business Entity Type 3,Working,Secondary / secondary special,-1134.0,406597.5,M,0.0833,-2120,0.0833,0.0833,0.0149,0.018801,0.0190,0.0193,0.0198,0.0247,0.0250,0.0252,2,0.9722,0.9722,0.9722,0.0690,2,0.0690,"Stone, brick",24700.5,-3648.0,0.0690,No,block of flats,Single / not married,1.0,House / apartment,Cash loans,0.0,0.0,0.0,0.0,0.0,202500.0,10,2.0,N,1.0,2.0,2.0,Unaccompanied,2.0,0,WEDNESDAY,Y,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,100003,0,NaN,0.622246,0.311267,-1188,1129500.0,-16765,Core staff,School,State servant,Higher education,-828.0,1293502.5,F,0.2917,-291,0.2917,0.2917,0.0714,0.003541,0.0549,0.0558,0.0554,0.0959,0.0968,0.0924,1,0.9851,0.9851,0.9851,0.0345,1,0.0345,Block,35698.5,-1186.0,0.0345,No,block of flats,Married,0.0,House / apartment,Cash loans,0.0,0.0,0.0,0.0,0.0,270000.0,11,0.0,N,2.0,1.0,1.0,Family,0.0,0,MONDAY,N,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,100004,0,0.729567,0.555912,NaN,-225,135000.0,-19046,Laborers,Government,Working,Secondary / secondary special,-815.0,135000.0,M,NaN,-2531,NaN,NaN,NaN,0.010032,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,2,NaN,NaN,6750.0,-4260.0,NaN,NaN,NaN,Single / not married,0.0,House / apartment,Revolving loans,0.0,0.0,0.0,0.0,0.0,67500.0,9,0.0,Y,1.0,0.0,0.0,Unaccompanied,0.0,0,MONDAY,Y,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,100006,0,NaN,0.650442,NaN,-3039,297000.0,-19005,Laborers,Business Entity Type 3,Working,Secondary / secondary special,-617.0,312682.5,F,NaN,-2437,NaN,NaN,NaN,0.008019,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,2,NaN,NaN,29686.5,-9833.0,NaN,NaN,NaN,Civil marriage,NaN,House / apartment,Cash loans,NaN,NaN,NaN,NaN,NaN,135000.0,17,0.0,N,2.0,2.0,2.0,Unaccompanied,0.0,0,WEDNESDAY,Y,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,100007,0,NaN,0.322738,NaN,-3038,513000.0,-19932,Core staff,Religion,Working,Secondary / secondary special,-1106.0,513000.0,M,NaN,-3458,NaN,NaN,NaN,0.028663,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,2,NaN,NaN,21865.5,-4311.0,NaN,NaN,NaN,Single / not married,0.0,House / apartment,Cash loans,0.0,0.0,0.0,0.0,0.0,121500.0,11,0.0,N,1.0,0.0,0.0,Unaccompanied,0.0,0,THURSDAY,Y,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1

# 3. Duplicate Check & Exact-Duplicate-Column Check

Validasi kualitas data paling dasar sebelum diproses lebih lanjut: cek duplikasi baris/ID, dan cek apakah ada kolom yang isinya 100% identik (bukan cuma berkorelasi tinggi — itu urusan 03_feature_engineering.ipynb).

In [71]:
df, dup_report = check_duplicates(df)
print("Duplicate check report:")
print(dup_report)

duplicate_column_groups = check_exact_duplicate_columns(df)
if duplicate_column_groups:
    print(f"\nDitemukan {len(duplicate_column_groups)} grup kolom identik:")
    for group in duplicate_column_groups:
        print(f"  {group}")
else:
    print("\nTidak ada kolom yang isinya 100% identik.")

Duplicate check report:
{'n_full_duplicate_rows': 0, 'n_id_duplicates': 0, 'action': 'Tidak ada duplikasi ditemukan.'}

Tidak ada kolom yang isinya 100% identik.


# 4. Anomali DAYS_EMPLOYED

Replace sentinel 365243 jadi NaN, buat flag untuk baris anomali yang bukan Pensioner, lalu impute median.

In [72]:
df = fix_days_employed_anomaly(df)
print("DAYS_EMPLOYED_ANOMALY value counts:")
print(df["DAYS_EMPLOYED_ANOMALY"].value_counts())
print(f"\nDAYS_EMPLOYED - missing setelah fix: {df['DAYS_EMPLOYED'].isnull().sum()}")

DAYS_EMPLOYED_ANOMALY value counts:
DAYS_EMPLOYED_ANOMALY
0    307489
1        22
Name: count, dtype: int64

DAYS_EMPLOYED - missing setelah fix: 0


# 5. Hidden Missing Value (XNA) & Imputasi Kategori Kecil

Replace placeholder "XNA" (bukan "Other") jadi NaN di tiga kolom, lalu imputasi CODE_GENDER dan NAME_FAMILY_STATUS (jumlahnya sangat kecil) dengan modus.

In [73]:
df = fix_hidden_missing_placeholders(df)
print("Missing setelah fix placeholder:")
print(df[["ORGANIZATION_TYPE", "CODE_GENDER", "NAME_FAMILY_STATUS"]].isnull().sum())

df = impute_gender_and_family_status(df)
print("\nMissing setelah imputasi CODE_GENDER & NAME_FAMILY_STATUS:")
print(df[["CODE_GENDER", "NAME_FAMILY_STATUS"]].isnull().sum())

Missing setelah fix placeholder:
ORGANIZATION_TYPE     55374
CODE_GENDER               4
NAME_FAMILY_STATUS        2
dtype: int64

Missing setelah imputasi CODE_GENDER & NAME_FAMILY_STATUS:
CODE_GENDER           0
NAME_FAMILY_STATUS    0
dtype: int64


# 6. Imputasi Grup Fitur Housing

Buat flag HAS_HOUSING_INFO, lalu impute median (numerik) dan kategori "Missing" (kategorikal) untuk grup fitur housing yang terbukti missing bersamaan (temuan 3.c EDA).

In [74]:
df = impute_housing_group(df)
print("HAS_HOUSING_INFO value counts:")
print(df["HAS_HOUSING_INFO"].value_counts())
print(f"\nTotal missing di grup housing setelah imputasi: {df[[c for c in df.columns if 'AREA' in c or 'FLOORS' in c]].isnull().sum().sum()}")

HAS_HOUSING_INFO value counts:
HAS_HOUSING_INFO
1    159080
0    148431
Name: count, dtype: int64

Total missing di grup housing setelah imputasi: 0


# 7. Imputasi EXT_SOURCE_1 & EXT_SOURCE_3

Median + flag _MISSING per kolom, karena bin "Missing" di perhitungan IV terbukti informatif.


In [75]:
df = impute_ext_source(df)
print("Missing EXT_SOURCE setelah imputasi:")
print(df[["EXT_SOURCE_1", "EXT_SOURCE_3"]].isnull().sum())
print("\nDistribusi flag missing:")
print(df[["EXT_SOURCE_1_MISSING", "EXT_SOURCE_3_MISSING"]].sum())

Missing EXT_SOURCE setelah imputasi:
EXT_SOURCE_1    0
EXT_SOURCE_3    0
dtype: int64

Distribusi flag missing:
EXT_SOURCE_1_MISSING    173378
EXT_SOURCE_3_MISSING     60965
dtype: int64


# 8. Imputasi AMT_REQ_CREDIT_BUREAU_*

Impute dengan 0, asumsi tidak ada rekaman inquiry = tidak ada inquiry bureau kredit.

In [76]:
df = impute_credit_bureau_inquiries(df)
bureau_cols = [c for c in df.columns if c.startswith("AMT_REQ_CREDIT_BUREAU")]
print(f"Missing di kolom AMT_REQ_CREDIT_BUREAU_* setelah imputasi: {df[bureau_cols].isnull().sum().sum()}")

Missing di kolom AMT_REQ_CREDIT_BUREAU_* setelah imputasi: 0


# 9. Verifikasi OCCUPATION_TYPE & ORGANIZATION_TYPE vs NAME_INCOME_TYPE

Ini cell eksploratif, tetap di notebook (bukan di src/) — bertujuan memastikan missing di dua kolom ini benar berasal dari klien yang tidak bekerja, sebelum menentukan nama kategori pengganti. Ini menjawab kekhawatiranmu sebelumnya soal risiko salah asumsi.

In [77]:
non_working_categories = ["Pensioner", "Unemployed", "Student"]

occupation_missing_mask = df["OCCUPATION_TYPE"].isnull()
occupation_crosstab = pd.crosstab(
    occupation_missing_mask, df["NAME_INCOME_TYPE"], normalize="index"
) * 100
print("Distribusi NAME_INCOME_TYPE - OCCUPATION_TYPE missing vs terisi (%):")
print(occupation_crosstab.T.round(2))

pct_occupation_non_working = df.loc[occupation_missing_mask, "NAME_INCOME_TYPE"].isin(non_working_categories).mean() * 100
print(f"\n{pct_occupation_non_working:.2f}% dari OCCUPATION_TYPE missing berasal dari kategori tidak bekerja")

organization_missing_mask = df["ORGANIZATION_TYPE"].isnull()
organization_crosstab = pd.crosstab(
    organization_missing_mask, df["NAME_INCOME_TYPE"], normalize="index"
) * 100
print("\nDistribusi NAME_INCOME_TYPE - ORGANIZATION_TYPE missing vs terisi (%):")
print(organization_crosstab.T.round(2))

pct_organization_non_working = df.loc[organization_missing_mask, "NAME_INCOME_TYPE"].isin(non_working_categories).mean() * 100
print(f"\n{pct_organization_non_working:.2f}% dari ORGANIZATION_TYPE missing berasal dari kategori tidak bekerja")

Distribusi NAME_INCOME_TYPE - OCCUPATION_TYPE missing vs terisi (%):
OCCUPATION_TYPE       False  True 
NAME_INCOME_TYPE                  
Businessman            0.00   0.00
Commercial associate  28.10  12.76
Maternity leave        0.00   0.00
Pensioner              0.00  57.43
State servant          8.49   3.93
Student                0.01   0.01
Unemployed             0.00   0.02
Working               63.40  25.85

57.46% dari OCCUPATION_TYPE missing berasal dari kategori tidak bekerja

Distribusi NAME_INCOME_TYPE - ORGANIZATION_TYPE missing vs terisi (%):
ORGANIZATION_TYPE     False  True 
NAME_INCOME_TYPE                  
Businessman            0.00   0.00
Commercial associate  28.40   0.00
Maternity leave        0.00   0.00
Pensioner              0.00  99.96
State servant          8.61   0.00
Student                0.01   0.00
Unemployed             0.00   0.04
Working               62.97   0.00

100.00% dari ORGANIZATION_TYPE missing berasal dari kategori tidak bekerja


# 10. Imputasi OCCUPATION_TYPE & ORGANIZATION_TYPE

Dipanggil setelah verifikasi di atas mengonfirmasi keputusan nama kategori.

In [78]:
df = impute_occupation_type(df, replacement_category="Not_Working")
df = impute_organization_type(df, replacement_category="Not_Working")

print("Missing setelah imputasi:")
print(df[["OCCUPATION_TYPE", "ORGANIZATION_TYPE"]].isnull().sum())

Missing setelah imputasi:
OCCUPATION_TYPE      0
ORGANIZATION_TYPE    0
dtype: int64


# 11. Log-Transform Fitur Finansial (Kolom Baru)

Buat kolom LOG_* untuk 4 fitur finansial sesuai outlier_treatment_decision.csv. Kolom asli tetap disimpan.

In [79]:
df = create_log_features(df)
log_cols = [c for c in df.columns if c.startswith("LOG_")]
print(f"Kolom log baru: {log_cols}")
df[log_cols].describe()

Kolom log baru: ['LOG_AMT_INCOME_TOTAL', 'LOG_AMT_CREDIT', 'LOG_AMT_ANNUITY', 'LOG_AMT_GOODS_PRICE']


,LOG_AMT_INCOME_TOTAL,LOG_AMT_CREDIT,LOG_AMT_ANNUITY,LOG_AMT_GOODS_PRICE
count,307511.000000,307511.000000,307511.000000,307511.000000
mean,11.909245,13.070108,10.067677,12.960539
std,0.488906,0.715193,0.545872,0.715197
min,10.152338,10.714440,7.388019,10.609082
25%,11.630717,12.506181,9.712630,12.382129
50%,11.899215,13.149068,10.122784,13.017005
75%,12.218500,13.603123,10.451522,13.429114
max,18.577685,15.214228,12.460818,15.214228


# 12. Rare Category Consolidation

Gabungkan kategori dengan n < 50 jadi "Rare" di tiga kolom kategorikal yang punya kategori sangat jarang.

In [80]:
before_counts = {col: df[col].nunique() for col in ["NAME_INCOME_TYPE", "ORGANIZATION_TYPE", "OCCUPATION_TYPE"]}

df = consolidate_rare_categories(df)

after_counts = {col: df[col].nunique() for col in ["NAME_INCOME_TYPE", "ORGANIZATION_TYPE", "OCCUPATION_TYPE"]}
print("Jumlah kategori unik sebelum vs setelah konsolidasi:")
for col in before_counts:
    print(f"  {col}: {before_counts[col]} -> {after_counts[col]}")

Jumlah kategori unik sebelum vs setelah konsolidasi:
  NAME_INCOME_TYPE: 8 -> 5
  ORGANIZATION_TYPE: 58 -> 57
  OCCUPATION_TYPE: 19 -> 19


# 13. Tipe Data Fitur Near-Zero Variance

Rapikan tipe data 18 fitur FLAG_DOCUMENT_*/FLAG_MOBIL/FLAG_CONT_MOBILE jadi int8. Tidak dibuang di sini — keputusan buang menunggu VarianceThreshold di 03_feature_engineering.ipynb.

In [81]:
df = ensure_near_zero_variance_dtype(df)
nzv_cols = [c for c in df.columns if c.startswith("FLAG_")]
print(df[nzv_cols].dtypes.value_counts())

int8     18
int64     8
str       2
Name: count, dtype: int64


# 14. Safety Net — Sisa Missing Value Apapun

Menangani sisa missing value yang belum tercakup fungsi spesifik manapun (misal DAYS_LAST_PHONE_CHANGE, 1 baris).

In [82]:
missing_before_safety_net = df.isnull().sum()
missing_before_safety_net = missing_before_safety_net[missing_before_safety_net > 0]
print("Sisa missing sebelum safety net:")
print(missing_before_safety_net)

df = impute_remaining_missing(df)

print(f"\nSisa missing setelah safety net: {df.isnull().sum().sum()}")

Sisa missing sebelum safety net:
EXT_SOURCE_2                 660
DAYS_LAST_PHONE_CHANGE         1
DEF_30_CNT_SOCIAL_CIRCLE    1021
CNT_FAM_MEMBERS                2
OBS_60_CNT_SOCIAL_CIRCLE    1021
OBS_30_CNT_SOCIAL_CIRCLE    1021
NAME_TYPE_SUITE             1292
DEF_60_CNT_SOCIAL_CIRCLE    1021
dtype: int64

Sisa missing setelah safety net: 0


# 15. Validasi Akhir

Dua validasi wajib: pastikan nol missing value, dan pastikan tipe data/kategori konsisten (FLAG_* cuma 0/1, CODE_GENDER cuma M/F, DAYS_* tetap ≤0).

In [83]:
validate_no_missing(df)
validate_dtypes_and_categories(df)

Validasi berhasil: tidak ada missing value tersisa.
Validasi berhasil: semua tipe data dan kategori konsisten.


True

# 16. Simpan Output Bersih

In [84]:
import os

OUTPUT_PATH = "../data/processed/application_train_clean.csv"
os.makedirs("../data/processed", exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

print(f"Shape akhir data bersih: {df.shape}")
print(f"Tersimpan di: {OUTPUT_PATH}")
print(f"\nKolom baru yang ditambahkan selama cleaning:")
new_cols = set(df.columns) - set(selected_features) - {"SK_ID_CURR", "TARGET"}
print(sorted(new_cols))

Shape akhir data bersih: (307511, 101)
Tersimpan di: ../data/processed/application_train_clean.csv

Kolom baru yang ditambahkan selama cleaning:
['DAYS_EMPLOYED_ANOMALY', 'EXT_SOURCE_1_MISSING', 'EXT_SOURCE_3_MISSING', 'HAS_HOUSING_INFO', 'LOG_AMT_ANNUITY', 'LOG_AMT_CREDIT', 'LOG_AMT_GOODS_PRICE', 'LOG_AMT_INCOME_TOTAL']
